In [1]:
from google.colab import drive
import os
import numpy as np
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from torchvision import models, transforms
from PIL import Image
import torch
from bokeh.plotting import figure, show, save
from bokeh.models import ColumnDataSource, HoverTool, CategoricalColorMapper
from bokeh.io import output_notebook
import base64
from bokeh.palettes import Category20
from datetime import datetime

In [3]:
# Enable Bokeh to display output in Jupyter/Colab
output_notebook()

# Mount Google Drive
drive.mount('/content/drive')

# Path to the main folder containing shape folders
main_folder_path = '/content/drive/MyDrive/sample shapes/shapes'

def load_and_process_images_with_resnet(folder_path):
    """
    Load images and extract features using a pre-trained ResNet model.
    """
    resnet = models.resnet18(pretrained=True)
    resnet.fc = torch.nn.Identity()  # Remove the classification layer
    resnet.eval()  # Set to evaluation mode

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    data = []
    labels = []
    image_paths = []

    for shape_folder in os.listdir(folder_path):
        shape_folder_path = os.path.join(folder_path, shape_folder)

        if os.path.isdir(shape_folder_path) and shape_folder.startswith("shape"):
            shape_label = shape_folder
            print(f"Loading and processing images from {shape_label}...")

            for file_name in os.listdir(shape_folder_path):
                if file_name.endswith(".png"):
                    file_path = os.path.join(shape_folder_path, file_name)
                    image = Image.open(file_path).convert('RGB')
                    image_tensor = preprocess(image).unsqueeze(0)  # Add batch dimension
                    with torch.no_grad():
                        features = resnet(image_tensor).numpy().flatten()
                    data.append(features)
                    labels.append(shape_label)
                    image_paths.append(file_path)
    return np.array(data), np.array(labels), image_paths

def convert_image_to_base64(image_path):
    """
    Convert an image to a base64 string.
    """
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode()
        return f"data:image/png;base64,{encoded_string}"

def apply_tsne_and_kmeans(data, n_clusters=21):
    """
    Apply t-SNE for dimensionality reduction and KMeans for clustering.
    """
    print("Applying t-SNE for dimensionality reduction...")
    tsne = TSNE(n_components=2, random_state=42)
    reduced_data = tsne.fit_transform(data)

    print("Applying KMeans for clustering...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(reduced_data)

    return reduced_data, cluster_labels

def save_interactive_plot(reduced_data, cluster_labels, labels, images, output_file_path):
    """
    Save an interactive plot with clustering results using Bokeh with base64 encoded images.
    """
    # Convert all image paths to base64 strings
    images_base64 = [convert_image_to_base64(image) for image in images]

    # Prepare the Bokeh data source
    source = ColumnDataSource(data=dict(
        x=reduced_data[:, 0],
        y=reduced_data[:, 1],
        cluster=[str(c) for c in cluster_labels],
        label=labels,
        image=images_base64
    ))

    # Define a color palette for the clusters
    unique_clusters = np.unique(cluster_labels)
    num_clusters = len(unique_clusters)
    palette = Category20[num_clusters] if num_clusters <= 20 else Category20[20]

    # Create a color mapper to map each cluster to a unique color
    color_mapper = CategoricalColorMapper(factors=[str(c) for c in unique_clusters], palette=palette)

    # Create a Bokeh figure with 'wheel_zoom' tool added
    p = figure(
        title="t-SNE with KMeans Clustering",
        tools="hover,pan,wheel_zoom,reset,save",
        active_scroll='wheel_zoom',
        tooltips="""
            <div>
                <strong>Cluster:</strong> @cluster <br>
                <strong>Label:</strong> @label <br>
                <img src="@image" height="64" style="float: left; margin: 0px 15px 15px 0px;"/>
            </div>
        """
    )

    # Add scatter plot with the color mapper
    p.scatter('x', 'y', size=5, source=source, legend_field='cluster',
              color={'field': 'cluster', 'transform': color_mapper})

    p.xaxis.axis_label = "t-SNE Dimension 1"
    p.yaxis.axis_label = "t-SNE Dimension 2"

    # Display the figure
    show(p)

    # Save the figure to an HTML file
    save(p, filename=output_file_path)

def main():
    # Load and process images
    data, labels, image_paths = load_and_process_images_with_resnet(main_folder_path)
    print(main_folder_path)

    if data.shape[0] == 0:
        print("No images found. Exiting.")
        return

    print(f"Loaded and processed {data.shape[0]} images from {len(np.unique(labels))} shapes.")

    # Apply t-SNE and KMeans
    reduced_data, cluster_labels = apply_tsne_and_kmeans(data, n_clusters=21)

    # Define output file path with current time and date
    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file_path = f"/content/drive/MyDrive/sample shapes/tsne_kmeans_plot_{current_time}.html"

    # Save the interactive plot
    save_interactive_plot(reduced_data, cluster_labels, labels, image_paths, output_file_path)

    print(f"Plot saved to {output_file_path}")


Mounted at /content/drive


In [ ]:
# Run the main function
main()

For incremental execution:

In [4]:
# Load and process images (this is the long process)
data, labels, image_paths = load_and_process_images_with_resnet(main_folder_path)
print(main_folder_path)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 106MB/s]


Loading and processing images from shape1...
Loading and processing images from shape2...
Loading and processing images from shape3...
Loading and processing images from shape4...
Loading and processing images from shape5...
Loading and processing images from shape6...
Loading and processing images from shape7...
Loading and processing images from shape8...
Loading and processing images from shape9...
Loading and processing images from shape10...
Loading and processing images from shape11...
Loading and processing images from shape12...
Loading and processing images from shape13...
Loading and processing images from shape14...
Loading and processing images from shape15...
Loading and processing images from shape16...
Loading and processing images from shape17...
Loading and processing images from shape18...
Loading and processing images from shape19...
Loading and processing images from shape20...
Loading and processing images from shape21...
Loading and processing images from shape22.

In [1]:
if data.shape[0] == 0:
        print("No images found. Exiting.")
        exit

print(f"Loaded and processed {data.shape[0]} images from {len(np.unique(labels))} shapes.")

NameError: name 'data' is not defined

In [ ]:
# Apply t-SNE and KMeans
number_of_clusters = 50
reduced_data, cluster_labels = apply_tsne_and_kmeans(data, n_clusters=number_of_clusters)
print(f"Applied t-SNE and KMeans with {number_of_clusters} clusters")

Applying t-SNE for dimensionality reduction...
Applying KMeans for clustering...
Applied t-SNE and KMeans with 50 clusters


In [ ]:
# Define output file path with current time and date
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file_path = f"/content/drive/MyDrive/sample shapes/clustering all shapes/tsne_kmeans_plot_{current_time}.html"

In [ ]:
# Save the interactive plot
save_interactive_plot(reduced_data, cluster_labels, labels, image_paths, output_file_path)

print(f"Plot saved to {output_file_path}")

Output hidden; open in https://colab.research.google.com to view.